In [13]:
import pandas as pd
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import torch.nn.functional as F
from pathlib import Path

# ============================================================
# DEVICE
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ============================================================
# LOAD PRETRAINED RESNET18
# ============================================================
model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
).to(device)

model.eval()

# ============================================================
# IMAGE TRANSFORM
# ============================================================
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

# ============================================================
# LOAD IMAGENET CLASS MAPPING
# ============================================================
import json
import urllib.request

def load_imagenet_mapping():
    url = "https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json"

    req = urllib.request.Request(
        url,
        headers={'User-Agent': 'Mozilla/5.0'}
    )

    with urllib.request.urlopen(req) as response:
        class_idx = json.loads(response.read().decode("utf-8"))

    return {v[0]: int(k) for k, v in class_idx.items()}

wnid_to_idx = load_imagenet_mapping()

print(f"Loaded {len(wnid_to_idx)} ImageNet classes.")


# ============================================================
# LOAD TEST SET
# ============================================================
df = pd.read_csv("lens_test.csv")

print(f"Found {len(df)} test images.")

# ============================================================
# EVALUATION
# ============================================================
top1_correct = 0
top5_correct = 0
total = len(df)

for i, row in df.iterrows():

    image_path = row["ae_image_path"]

    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)

    # Folder name is the ImageNet class
    class_name = Path(image_path).parent.name

    if class_name not in wnid_to_idx:
        continue

    target = wnid_to_idx[class_name]

    with torch.no_grad():

        output = model(image)

        # ---------------- Top-1 ----------------
        prediction = output.argmax(dim=1).item()

        if prediction == target:
            top1_correct += 1

        # ---------------- Top-5 ----------------
        top5 = torch.topk(output, k=5, dim=1).indices.squeeze(0)

        if target in top5:
            top5_correct += 1

    if (i + 1) % 50 == 0:
        print(f"Processed {i+1}/{total} images")

# ============================================================
# RESULTS
# ============================================================
top1_acc = 100 * top1_correct / total
top5_acc = 100 * top5_correct / total

print("\n===============================")
print("Baseline 1 : Auto Exposure")
print("===============================")
print(f"Images Evaluated : {total}")
print(f"Top-1 Accuracy   : {top1_acc:.2f}%")
print(f"Top-5 Accuracy   : {top5_acc:.2f}%")

Using device: cuda
Loaded 1000 ImageNet classes.
Found 600 test images.
Processed 50/600 images
Processed 100/600 images
Processed 150/600 images
Processed 200/600 images
Processed 250/600 images
Processed 300/600 images
Processed 350/600 images
Processed 400/600 images
Processed 450/600 images
Processed 500/600 images
Processed 550/600 images
Processed 600/600 images

Baseline 1 : Auto Exposure
Images Evaluated : 600
Top-1 Accuracy   : 4.33%
Top-5 Accuracy   : 12.50%


In [9]:
import random

sample = df.sample(5, random_state=42)
for _, row in sample.iterrows():
    from pathlib import Path
    p = Path(row["ae_image_path"])
    class_name = p.parent.name
    print(class_name, class_name in wnid_to_idx, wnid_to_idx.get(class_name))

n03930313 True 716
n01644900 True 32
n01983481 True 122
n01774384 True 75
n03126707 True 517


In [2]:
import json
import urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

# ============================================================
# DEVICE
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ============================================================
# LOAD PRETRAINED RESNET18
# ============================================================
model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
).to(device)

model.eval()

# ============================================================
# IMAGE TRANSFORM
# ============================================================
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

# ============================================================
# LOAD IMAGENET CLASS MAP
# ============================================================
def load_imagenet_mapping():

    url = "https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json"

    req = urllib.request.Request(
        url,
        headers={'User-Agent':'Mozilla/5.0'}
    )

    with urllib.request.urlopen(req) as response:
        class_idx = json.loads(response.read().decode("utf-8"))

    return {v[0]: int(k) for k,v in class_idx.items()}

wnid_to_idx = load_imagenet_mapping()

print(f"Loaded {len(wnid_to_idx)} ImageNet classes.")

# ============================================================
# LOAD TEST CSV
# ============================================================
df = pd.read_csv("lens_test.csv")

print(f"Found {len(df)} test images.")

# ============================================================
# EVALUATION
# ============================================================
top1_correct = 0
top5_correct = 0

total = len(df)

for i, row in df.iterrows():

    ae_path = Path(row["ae_image_path"])

    # Oracle parameter (convert back to 1-27)
    confidence_vector = np.fromstring(row['confidence_vector'].strip("[]"), sep=" ")
    oracle_param = int(torch.argmax(torch.tensor(confidence_vector))) + 1
    # print(oracle_param)

    # oracle_param = int(row["best_param_index"]) + 1

    light_env = row["light_environment"]

    class_name = ae_path.parent.name
    filename = ae_path.name

    # Build oracle image path
    oracle_path = (
      ae_path.parents[4]
      / "param_control"
      / light_env
      / f"param_{oracle_param}"
      / class_name
      / filename
  )


    if not oracle_path.exists():
        continue

    image = Image.open(oracle_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)

    target = wnid_to_idx[class_name]

    with torch.no_grad():

        output = model(image)

        # Top-1
        prediction = output.argmax(dim=1).item()

        if prediction == target:
            top1_correct += 1

        # Top-5
        top5 = torch.topk(output, 5, dim=1).indices.squeeze(0)

        if target in top5:
            top5_correct += 1

    if (i + 1) % 50 == 0:
        print(f"Processed {i+1}/{total}")

# ============================================================
# RESULTS
# ============================================================
top1_acc = 100 * top1_correct / total
top5_acc = 100 * top5_correct / total

print("\n===============================")
print("Baseline 2 : Oracle Parameters")
print("===============================")
print(f"Images Evaluated : {total}")
print(f"Top-1 Accuracy   : {top1_acc:.2f}%")
print(f"Top-5 Accuracy   : {top5_acc:.2f}%")

Using device: cuda
Loaded 1000 ImageNet classes.
Found 600 test images.
Processed 50/600
Processed 100/600
Processed 150/600
Processed 200/600
Processed 250/600
Processed 300/600
Processed 350/600
Processed 400/600
Processed 450/600
Processed 500/600
Processed 550/600
Processed 600/600

Baseline 2 : Oracle Parameters
Images Evaluated : 600
Top-1 Accuracy   : 34.00%
Top-5 Accuracy   : 56.67%


In [6]:
import json
import urllib.request
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

# ============================================================
# DEVICE
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# ============================================================
# FEATURE EXTRACTOR (from utils.py) - frozen resnet18, fc removed
# ============================================================
_resnet_backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT).to(device)
feature_extractor = nn.Sequential(*list(_resnet_backbone.children())[:-1]).to(device)
feature_extractor.eval()

for param in feature_extractor.parameters():
    param.requires_grad = False


def extract_features(image_tensor):
    """
    Extracts lighting-aware features from an AE image tensor (3, H, W).
    Returns a 1D tensor of shape (586,): 74 handcrafted + 512 deep features.
    """
    r, g, b = image_tensor[0], image_tensor[1], image_tensor[2]
    _, H, W = image_tensor.shape
    features = []

    # ── 1. RGB spatial histogram (60 dims) ─────────────────────────────
    grid = 2
    h_step, w_step = H // grid, W // grid
    for row in range(grid):
        for col in range(grid):
            ys, ye = row * h_step, (row + 1) * h_step
            xs, xe = col * w_step, (col + 1) * w_step
            n_pixels = h_step * w_step
            for channel in [r, g, b]:
                block = channel[ys:ye, xs:xe]
                hist = torch.histc(block, bins=5, min=0.0, max=1.0)
                features.append(hist / n_pixels)

    # ── 2. Per-channel global stats (9 dims) ────────────────────────────
    for channel in [r, g, b]:
        ch_mean = channel.mean()
        ch_std = channel.std()
        clipping = ((channel > 0.95) | (channel < 0.05)).float().mean()
        features.append(torch.stack([ch_mean, ch_std, clipping]))

    # ── 3. Local brightness contrast (5 dims) ────────────────────────────
    gray = 0.2989 * r + 0.5870 * g + 0.1140 * b
    block_means = []
    for row in range(grid):
        for col in range(grid):
            ys, ye = row * h_step, (row + 1) * h_step
            xs, xe = col * w_step, (col + 1) * w_step
            block_means.append(gray[ys:ye, xs:xe].mean())
    block_means_t = torch.stack(block_means)
    global_contrast = block_means_t.std()
    features.append(block_means_t)
    features.append(global_contrast.unsqueeze(0))

    # ── 4. Deep resnet18 features (512 dims) ─────────────────────────────
    with torch.no_grad():
        image_batch = image_tensor.unsqueeze(0).to(device)
        deep_features = feature_extractor(image_batch)
        deep_features = deep_features.flatten()

    handcrafted = torch.cat([f.flatten() for f in features])
    final = torch.cat([handcrafted, deep_features.cpu()])

    return final


# ============================================================
# LENS PREDICTOR MODEL (from models.py)
# ============================================================
class LensPredictorNetwork(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 27)
        )

    def forward(self, x):
        return self.network(x)


# ============================================================
# LOAD TRAINED LENS PREDICTOR
# ============================================================
INPUT_DIM = 586
MODEL_PATH = "resnet_histogram.pth"   # change to your actual saved checkpoint filename

lens_model = LensPredictorNetwork(INPUT_DIM).to(device)

lens_model.load_state_dict(
    torch.load(MODEL_PATH, map_location=device)
)

lens_model.eval()

print("Lens predictor loaded.")


# ============================================================
# LOAD IMAGENET RESNET18 (final evaluation classifier)
# ============================================================
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT).to(device)
resnet.eval()

print("ImageNet ResNet loaded.")


# ============================================================
# IMAGE TRANSFORM
# ============================================================
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ============================================================
# IMAGENET CLASS MAP
# ============================================================
def load_mapping():
    url = "https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json"
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as response:
        data = json.loads(response.read().decode("utf-8"))
    return {v[0]: int(k) for k, v in data.items()}


wnid_to_idx = load_mapping()


# ============================================================
# LOAD TEST CSV - TAKE FIRST 600 IMAGES
# ============================================================
df = pd.read_csv("lens_test.csv")
df = df.iloc[:600].reset_index(drop=True)

print(f"Evaluating on {len(df)} test samples.")


# ============================================================
# EVALUATION
# ============================================================
top1_correct = 0
top5_correct = 0
total = len(df)

for i, row in df.iterrows():

    ae_path = Path(row["ae_image_path"])

    # ------------------------
    # Load AE image
    # ------------------------
    ae_image = Image.open(ae_path).convert("RGB")
    ae_tensor = transform(ae_image)

    # ------------------------
    # Extract lighting-aware features (586-dim, per utils.py)
    # ------------------------
    feat = extract_features(ae_tensor).unsqueeze(0).to(device)

    # ------------------------
    # Predict lens parameter (1-27)
    # ------------------------
    with torch.no_grad():
        output = lens_model(feat)
        predicted_param = output.argmax(dim=1).item() + 1

    # ------------------------
    # Build predicted parameter image path
    # ------------------------
    class_name = ae_path.parent.name
    filename = ae_path.name

    param_path = (
        ae_path.parents[4]
        / "param_control"
        / row["light_environment"]
        / f"param_{predicted_param}"
        / class_name
        / filename
    )

    if not param_path.exists():
        continue

    # ------------------------
    # Load predicted parameter image
    # ------------------------
    img = Image.open(param_path).convert("RGB")
    img = transform(img).unsqueeze(0).to(device)

    target = wnid_to_idx[class_name]

    # ------------------------
    # ImageNet prediction (Top-1 / Top-5)
    # ------------------------
    with torch.no_grad():
        logits = resnet(img)
        pred = logits.argmax(dim=1).item()

        if pred == target:
            top1_correct += 1

        top5 = torch.topk(logits, k=5, dim=1).indices.squeeze(0)
        if target in top5:
            top5_correct += 1

    if (i + 1) % 50 == 0:
        print(f"Processed {i+1}/{total}")


# ============================================================
# RESULTS
# ============================================================
top1_acc = 100 * top1_correct / total
top5_acc = 100 * top5_correct / total

print("\n===============================")
print("Trained LensPredictorNetwork Evaluation")
print("===============================")
print(f"Images Evaluated : {total}")
print(f"Top-1 Accuracy   : {top1_acc:.2f}%")
print(f"Top-5 Accuracy   : {top5_acc:.2f}%")

Using device: cuda
Lens predictor loaded.
ImageNet ResNet loaded.
Evaluating on 600 test samples.
Processed 50/600
Processed 100/600
Processed 150/600
Processed 200/600
Processed 250/600
Processed 300/600
Processed 350/600
Processed 400/600
Processed 450/600
Processed 500/600
Processed 550/600
Processed 600/600

Trained LensPredictorNetwork Evaluation
Images Evaluated : 600
Top-1 Accuracy   : 15.83%
Top-5 Accuracy   : 32.50%
